# 8.13 同名 LoRA 參數代表同樣更新幅度嗎？

# 第 8 章：風格、個性與指令遵循

前置：第7章SFT。把風格與內容分開想：想像同一位老師可以說得生動，也可以精準遵循格式。本章先用人工編寫、內容固定的短回答校準目標；LoRA是後半支線。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：原畫不重畫：加一張低rank修正圖層**

基模仍佔空間；只是微調參數變少。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/lora.svg")))

**先想一想：**rank加倍但alpha固定，縮放因子會變大嗎？

rank和alpha共同決定更新縮放。有些工具使用其他convention，同名參數不保證相同delta；存adapter時要寫公式。

**把直覺寫成數學：**本教材縮放alpha/r；r固定時alpha加倍，ΔW加倍。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.alignment import LoRALinear

layer = LoRALinear(nn.Linear(4, 3), rank=2, alpha=2)
with torch.no_grad():
    layer.b.fill_(1.0)
delta1 = layer.merged_weight() - layer.base.weight
layer.alpha = 4
delta2 = layer.merged_weight() - layer.base.weight
print((delta2 - 2 * delta1).abs().max().item())
assert torch.allclose(delta2, 2 * delta1, atol=1e-6)

**如何讀結果：**相同rank/alpha但A/B不同也會有不同更新；記錄完整adapter狀態。

**只改一件事：**只改rank計算理論縮放，不混用其他作者公式。
